# Multimodal RAG Pipeline (LangChain)
Parse the PDF with Docling (text + tables + VLM image captions), index it in a LangChain `Chroma` vector store, and answer questions with a local Ollama Qwen model.

In [ ]:
from pathlib import Path
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, smolvlm_picture_description
from docling.chunking import HybridChunker
from docling_core.types.doc import PictureItem

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama

In [ ]:
PDF_PATH = "datasets/Generative AI System Design Interview{Ali Aminian_ Hao Sheng}(2024 November 16, ByteByteGo, Inc){111104690} libgen.li.pdf"

# Set a (start, end) 1-based page range to test on a subset (fast). None = whole 351-page book.
PAGE_RANGE = (1, 30)

## 1. Ingestion — parse, chunk, build LangChain Documents

In [ ]:
# Multimodal converter: keep images, caption them with a local VLM, parse tables
po = PdfPipelineOptions()
po.generate_picture_images = True                       # keep image bitmaps
po.images_scale = 1.0                                   # 1.0 = faster VLM; raise to 2.0 for sharper crops
po.do_picture_description = True                        # VLM captions for figures (set False for a big speedup)
po.picture_description_options = smolvlm_picture_description  # local SmolVLM (swap for PictureDescriptionApiOptions for speed)
po.do_table_structure = True                           # parse table structure

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=po)}
)

In [ ]:
# Parse the PDF, caching the result so the slow Docling+SmolVLM pass runs only once.
# Delete the cache file (or change PAGE_RANGE) to force a re-parse.
from docling_core.types.doc import DoclingDocument

tag = f"{PAGE_RANGE[0]}-{PAGE_RANGE[1]}" if PAGE_RANGE else "full"
cache = Path(f"doc_cache_{tag}.json")

if cache.exists():
    doc = DoclingDocument.load_from_json(cache)
    print(f"Loaded cached doc from {cache}")
else:
    result = converter.convert(PDF_PATH, page_range=PAGE_RANGE) if PAGE_RANGE else converter.convert(PDF_PATH)
    doc = result.document
    doc.save_as_json(cache)
    print(f"Parsed and cached -> {cache}")

In [ ]:
# Text + table chunks -> LangChain Documents (HybridChunker serializes tables into text)
docs = [
    Document(
        page_content=c.text,
        metadata={"type": "text", "headings": " > ".join(c.meta.headings or [])},
    )
    for c in HybridChunker().chunk(doc)
]
print(f"{len(docs)} text/table documents")

In [ ]:
# Image documents: save each figure + add its VLM caption as a Document
Path("images").mkdir(exist_ok=True)
n_img = 0
for item, _ in doc.iterate_items():
    if not isinstance(item, PictureItem):
        continue
    img = item.get_image(doc)
    if img is None:
        continue
    path = f"images/fig_{n_img}.png"
    img.save(path)
    n_img += 1
    caption = next((a.text for a in item.annotations if getattr(a, "text", None)), "") or item.caption_text(doc)
    if caption:
        docs.append(Document(page_content=caption, metadata={"type": "image", "image_path": path}))

print(f"{n_img} images, {sum(d.metadata['type'] == 'image' for d in docs)} captioned -> {len(docs)} total documents")

## 2. Index in a LangChain Chroma vector store

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma(
    collection_name="genai_book",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)
vectorstore.reset_collection()      # clear any prior run so docs don't accumulate/duplicate
vectorstore.add_documents(docs)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"Indexed {vectorstore._collection.count()} documents")

## 3. RAG chain — retrieve + generate with local Ollama Qwen

In [ ]:
llm = ChatOllama(model="qwen2.5:14b", temperature=0)

prompt = ChatPromptTemplate.from_template(
    "Answer the question using ONLY the context below. "
    "If the answer isn't in the context, say you don't know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)

def format_docs(retrieved):
    return "\n\n".join(f"[{d.metadata.get('type', 'text')}] {d.page_content}" for d in retrieved)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# Try it
query = "How would you design a recommendation system?"
print(rag_chain.invoke(query))

print("\n--- sources ---")
for d in retriever.invoke(query):
    print(f"[{d.metadata.get('type', 'text')}] {d.metadata.get('image_path', d.metadata.get('headings', ''))}")

## 4. Evaluation — sample questions (easy / medium / complex)

In [ ]:
questions = {
    "easy": [
        "What is the difference between a discriminative and a generative model?",
        "What are common data sources used to train a recommendation system?",
        "What is an embedding?",
    ],
    "medium": [
        "How would you frame recommendation as a machine learning problem, and what model type fits?",
        "What evaluation metrics would you use offline vs. online for a recommendation system?",
        "How do you handle the cold-start problem for new users or new items?",
    ],
    "complex": [
        "Design an end-to-end image-generation system: data, model choice (GAN vs. diffusion), training, evaluation, and serving constraints.",
        "How would you design a retrieval + ranking pipeline at scale, and where do latency/quality trade-offs force compromises?",
        "How would you incorporate real-time user feedback into a recommendation system without destabilizing the model?",
    ],
}

for tier, qs in questions.items():
    for q in qs:
        print(f"\n{'='*80}\n[{tier.upper()}] {q}\n{'-'*80}")
        print(rag_chain.invoke(q))

## 5. Conversational RAG with memory
Multi-turn chat: a contextualize step rewrites follow-ups into standalone questions using chat history, then the answer step responds from retrieved context. History is kept per `session_id`.

In [ ]:
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import MessagesPlaceholder

# Step 1: rewrite a follow-up into a standalone question using chat history
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given the chat history and the latest user question, rewrite it as a standalone "
               "question. Do NOT answer it; just reformulate if needed, otherwise return it as-is."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_prompt)

# Step 2: answer from the retrieved context ({context} is filled by the chain)
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the context below. If the answer isn't in the context, say you "
               "don't know.\n\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
qa_chain = create_stuff_documents_chain(llm, qa_prompt)
conv_rag = create_retrieval_chain(history_aware_retriever, qa_chain)

In [ ]:
# Wrap with per-session memory (in-session; cleared on kernel restart)
store = {}

def get_history(session_id):
    return store.setdefault(session_id, InMemoryChatMessageHistory())

chat_rag = RunnableWithMessageHistory(
    conv_rag,
    get_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

def chat(q, session_id="default"):
    out = chat_rag.invoke({"input": q}, config={"configurable": {"session_id": session_id}})
    return out["answer"]

In [ ]:
# Demo: successive turns — the 2nd question relies on memory ("it" = recommendation system)
print("Q1:", "How would you design a recommendation system?")
print(chat("How would you design a recommendation system?"), "\n")

print("Q2:", "What metrics would I use to evaluate it?")
print(chat("What metrics would I use to evaluate it?"), "\n")

# Inspect accumulated conversation
print("--- history ---")
for m in store["default"].messages:
    print(f"{m.type}: {m.content[:80]}")